# Лабораторная 1: декодерный трансформер для предсказания следующего токена

## Что нужно знать до старта
Перед началом лабораторной полезно открыть:
- [../README.md](./README.md)
- [guides/00_autoregression_prerequisites.md](./guides/00_autoregression_prerequisites.md)
- [guides/01_decoder_only_toy_walkthrough.md](./guides/01_decoder_only_toy_walkthrough.md)
- [../theory/theory.md](../theory/theory.md)

Это первая лабораторная блока `04-Autoregression` и Шаг 7 общего курса.


## Выбор среды выполнения

Рекомендуемый стартовый режим: `auto`.

Если нужна полностью воспроизводимая проверка на центральном процессоре, выберите `local-cpu` и выполните `Restart & Run All`.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _detect_notebook_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Интуиция задачи без формул

Модель предсказывает следующий токен, опираясь только на прошлые позиции. Для этого в блоке внимания используется причинная маска (causal mask).


## Как проходить работу

Маршрут строго фиксирован: данные → маска → блок декодера → обучение → генерация → диагностика внимания.


## Контракт данных

В этой лабораторной используются последовательности фиксированной максимальной длины.

- Вход `X`: идентификаторы токенов без последнего шага, форма `(N, T)`.
- Цель `Y`: те же последовательности, сдвинутые на один шаг влево, форма `(N, T)`.
- Дополнение пустыми позициями задаётся токеном `PAD = 0`.

Разбиение на обучение, проверку и тест выполняется строго индексами без перемешивания.


## Таблица ключевых форм

| Сущность | Смысл | Форма |
|---|---|---|
| `X` | входные токены | `(N, T)` |
| `Y` | цели следующего токена | `(N, T)` |
| `mask` | причинная маска | `(T, T)` |
| `attention_scores` | веса внимания | `(N, H, T, T)` |
| `y_pred` | вероятности токенов | `(N, T, V)` |


## Ручной пример

Пусть последовательность имеет вид:

```text
<BOS> A B C <EOS> <PAD> <PAD>
```

Тогда пара для обучения следующему токену строится так:

```text
X: <BOS> A B C <EOS> <PAD>
Y: A B C <EOS> <PAD> <PAD>
```

То есть на каждом шаге модель должна восстановить токен, стоящий на одну позицию правее.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 13
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
TOKEN_A = 3
TOKEN_B = 4
TOKEN_C = 5
TOKEN_D = 6

ID_TO_TOKEN = {
    PAD_ID: '<PAD>',
    BOS_ID: '<BOS>',
    EOS_ID: '<EOS>',
    TOKEN_A: 'A',
    TOKEN_B: 'B',
    TOKEN_C: 'C',
    TOKEN_D: 'D',
}
VOCAB_SIZE = len(ID_TO_TOKEN)
MAX_SEQ_LEN = 18
TRAIN_SAMPLES = 4096
VAL_SAMPLES = 512
TEST_SAMPLES = 512
TOTAL_SAMPLES = TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
BATCH_SIZE = 64
EPOCHS = 8

plt.style.use('default')
keras.utils.set_random_seed(SEED)


## Шаг 1: подготовка детерминированных данных


In [ ]:
def build_synthetic_corpus(num_samples, max_seq_len=MAX_SEQ_LEN, seed=SEED):
    """Создаёт детерминированный синтетический корпус для лабораторной.

    Args:
      num_samples: Число последовательностей в корпусе.
      max_seq_len: Максимальная длина последовательности с учётом дополнения.
      seed: Зерно случайности для воспроизводимости.

    Returns:
      Кортеж `(X, Y)`, где обе матрицы имеют форму `(num_samples, max_seq_len - 1)`.

    Raises:
      ValueError: Если `max_seq_len` слишком мал для корректной сборки последовательностей.
    """
    if max_seq_len < 8:
        raise ValueError('max_seq_len должен быть не меньше 8.')

    rng = np.random.default_rng(seed)
    data = np.full((num_samples, max_seq_len), PAD_ID, dtype=np.int32)

    transition = {
        TOKEN_A: TOKEN_B,
        TOKEN_B: TOKEN_C,
        TOKEN_C: TOKEN_A,
        TOKEN_D: TOKEN_B,
    }
    start_tokens = np.array([TOKEN_A, TOKEN_B, TOKEN_C, TOKEN_D], dtype=np.int32)
    payload_lengths = rng.integers(6, 12, size=num_samples)

    for row_index, payload_len in enumerate(payload_lengths):
        current = int(start_tokens[rng.integers(0, len(start_tokens))])
        payload = [current]

        # Переходы полностью детерминированы: следующий токен однозначно задаётся текущим.
        for _ in range(payload_len - 1):
            current = transition[current]
            payload.append(current)

        sequence = [BOS_ID, *payload, EOS_ID]
        data[row_index, : len(sequence)] = np.asarray(sequence, dtype=np.int32)

    x_matrix = data[:, :-1]
    y_matrix = data[:, 1:]
    return x_matrix, y_matrix


X_all, y_all = build_synthetic_corpus(TOTAL_SAMPLES, seed=SEED)
X_train = X_all[:TRAIN_SAMPLES]
y_train = y_all[:TRAIN_SAMPLES]
X_val = X_all[TRAIN_SAMPLES : TRAIN_SAMPLES + VAL_SAMPLES]
y_val = y_all[TRAIN_SAMPLES : TRAIN_SAMPLES + VAL_SAMPLES]
X_test = X_all[TRAIN_SAMPLES + VAL_SAMPLES :]
y_test = y_all[TRAIN_SAMPLES + VAL_SAMPLES :]


In [ ]:
def ids_to_text(token_ids):
    """Преобразует список идентификаторов токенов в читаемую строку.

    Args:
      token_ids: Последовательность целых идентификаторов.

    Returns:
      Строка с токенами в порядке следования.
    """
    return ' '.join(ID_TO_TOKEN.get(int(token), f'<UNK:{int(token)}>') for token in token_ids)


k = 8
x_a, y_a = build_synthetic_corpus(k, seed=SEED + 101)
x_b, y_b = build_synthetic_corpus(k, seed=SEED + 101)

assert np.array_equal(x_a, x_b), 'Первые k примеров не воспроизводятся при одинаковом seed.'
assert np.array_equal(y_a, y_b), 'Целевые значения не воспроизводятся при одинаковом seed.'

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('Пример X[0]:', ids_to_text(X_train[0]))
print('Пример y[0]:', ids_to_text(y_train[0]))


## Шаг 2: причинная маска и её проверка


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Args:
      seq_len: Длина последовательности.

    Returns:
      Булев тензор формы `(seq_len, seq_len)`.

    Raises:
      ValueError: Если `seq_len` не является положительным числом.
    """
    if seq_len <= 0:
        raise ValueError('seq_len должен быть положительным.')
    return tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0)


def assert_lower_triangular(mask):
    """Проверяет, что маска имеет нижнетреугольную структуру.

    Args:
      mask: Булева матрица формы `(T, T)`.

    Raises:
      AssertionError: Если верхний треугольник содержит разрешающие значения.
    """
    array_mask = np.asarray(mask).astype(np.int32)
    upper = np.triu(array_mask, k=1)
    assert upper.sum() == 0, 'Верхний треугольник должен быть полностью закрыт.'


mask_example = build_causal_mask(8).numpy()
assert mask_example.shape == (8, 8)
assert_lower_triangular(mask_example)
print(mask_example.astype(np.int32))


## Шаг 3: декодерный блок и метрики


In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """Складывает векторы токенов и позиций в едином представлении.

    Args:
      maxlen: Максимальная длина последовательности.
      vocab_size: Размер словаря.
      embed_dim: Размер скрытого представления.
    """

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует таблицы токенных и позиционных векторов.

        Args:
          maxlen: Максимальная длина последовательности.
          vocab_size: Размер словаря.
          embed_dim: Размер векторного представления.
          **kwargs: Дополнительные аргументы базового класса.
        """
        super().__init__(**kwargs)
        self.maxlen = maxlen
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Вычисляет сумму токенных и позиционных векторов.

        Args:
          inputs: Матрица идентификаторов токенов формы `(batch, time)`.

        Returns:
          Тензор формы `(batch, time, embed_dim)`.
        """
        positions = tf.range(start=0, limit=tf.shape(inputs)[-1], delta=1)
        token_vectors = self.token_embedding(inputs)
        position_vectors = self.position_embedding(positions)
        return token_vectors + position_vectors

    def compute_mask(self, inputs, mask=None):
        """Переадресует маску непустых токенов из слоя токенных векторов.

        Args:
          inputs: Матрица идентификаторов токенов.
          mask: Входная маска базового уровня.

        Returns:
          Булева маска формы `(batch, time)`.
        """
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской.

    Args:
      embed_dim: Размер скрытого представления.
      num_heads: Число голов внимания.
      ff_dim: Ширина промежуточного слоя позиционно-независимой сети.
      rate: Доля отключаемых нейронов в регуляризации.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои блока.

        Args:
          embed_dim: Размер скрытого представления.
          num_heads: Число голов внимания.
          ff_dim: Ширина промежуточного слоя позициионно-независимой сети.
          rate: Доля отключаемых нейронов в регуляризации.
          **kwargs: Дополнительные аргументы базового класса.

        Raises:
          ValueError: Если `embed_dim` не делится на `num_heads`.
        """
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')

        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ],
            name='позиционно_независимая_сеть',
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через внимание и позиционно-независимую сеть.

        Args:
          inputs: Тензор формы `(batch, time, embed_dim)`.
          padding_mask: Булева маска непустых позиций формы `(batch, time)`.
          training: Признак режима обучения.
          return_attention_scores: Нужно ли вернуть веса внимания.

        Returns:
          Если `return_attention_scores=False`, возвращается тензор признаков.
          Если `return_attention_scores=True`, возвращается кортеж
          `(признаки, веса_внимания)`.
        """
        seq_len = tf.shape(inputs)[1]
        causal_mask = build_causal_mask(seq_len)[tf.newaxis, :, :]

        if padding_mask is None:
            attention_mask = causal_mask
        else:
            # Ключи и запросы одновременно ограничиваются непустыми позициями.
            key_mask = tf.cast(padding_mask[:, tf.newaxis, :], tf.bool)
            query_mask = tf.cast(padding_mask[:, :, tf.newaxis], tf.bool)
            attention_mask = causal_mask & key_mask & query_mask

        attention_output, attention_scores = self.self_attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            return_attention_scores=True,
            training=training,
        )

        x = self.norm_1(inputs + self.dropout_1(attention_output, training=training))
        y = self.ffn(x)
        output = self.norm_2(x + self.dropout_2(y, training=training))

        if return_attention_scores:
            return output, attention_scores
        return output

    def compute_mask(self, inputs, mask=None):
        """Передаёт маску дальше без изменений.

        Args:
          inputs: Входной тензор признаков.
          mask: Текущая маска.

        Returns:
          Маска, совпадающая с входной.
        """
        return mask


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию только по непустым позициям.

    Args:
      y_true: Истинные идентификаторы токенов.
      y_pred: Предсказанные вероятности токенов.

    Returns:
      Среднее значение функции потерь по непустым позициям.
    """
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    masked = per_token * mask
    return tf.reduce_sum(masked) / tf.reduce_sum(mask)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность только по непустым позициям.

    Args:
      y_true: Истинные идентификаторы токенов.
      y_pred: Предсказанные вероятности токенов.

    Returns:
      Доля верных предсказаний по непустым позициям.
    """
    predicted = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, predicted), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


def perplexity_from_loss(loss_value):
    """Преобразует среднюю перекрёстную энтропию в перплексию.

    Args:
      loss_value: Среднее значение функции потерь.

    Returns:
      Значение перплексии.
    """
    return float(np.exp(loss_value))


## Шаг 4: сборка и обучение модели


In [ ]:
keras.utils.set_random_seed(SEED)

token_ids = keras.Input(shape=(MAX_SEQ_LEN - 1,), dtype='int32', name='token_ids')
padding_mask = layers.Lambda(lambda t: tf.not_equal(t, PAD_ID), name='padding_mask')(token_ids)

embedding_layer = TokenAndPositionEmbedding(
    maxlen=MAX_SEQ_LEN - 1,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    name='token_and_position',
)
decoder_layer = CausalDecoderBlock(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    rate=0.0,
    name='causal_decoder',
)

x = embedding_layer(token_ids)
x = decoder_layer(x, padding_mask=padding_mask)
next_token_probs = layers.Dense(VOCAB_SIZE, activation='softmax', name='next_token_probs')(x)

model = keras.Model(inputs=token_ids, outputs=next_token_probs, name='decoder_only_toy_model')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=masked_sparse_crossentropy,
    metrics=[masked_token_accuracy],
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    verbose=2,
)

model.summary()


In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='обучение')
plt.plot(history.history['val_loss'], label='проверка')
plt.title('Перекрёстная энтропия по эпохам')
plt.xlabel('Эпоха')
plt.ylabel('loss')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in history.history['loss']]
val_ppl = [perplexity_from_loss(v) for v in history.history['val_loss']]
plt.plot(train_ppl, label='обучение')
plt.plot(val_ppl, label='проверка')
plt.title('Перплексия по эпохам')
plt.xlabel('Эпоха')
plt.ylabel('perplexity')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()


In [ ]:
test_loss, test_token_acc = model.evaluate(X_test, y_test, verbose=0)
test_perplexity = perplexity_from_loss(test_loss)

print(f'test_loss            : {test_loss:.4f}')
print(f'test_token_accuracy  : {test_token_acc:.4f}')
print(f'test_perplexity      : {test_perplexity:.4f}')

assert test_token_acc >= 0.97, 'Токенная точность ниже целевого порога 0.97.'
assert test_perplexity <= 1.30, 'Перплексия выше целевого порога 1.30.'


## Шаг 5: детерминированная генерация


In [ ]:
def generate_greedy(model, prompt_ids, max_new_tokens, max_seq_len=MAX_SEQ_LEN - 1):
    """Генерирует продолжение в детерминированном режиме argmax.

    Args:
      model: Обученная модель предсказания следующего токена.
      prompt_ids: Начальная последовательность идентификаторов токенов.
      max_new_tokens: Максимум новых шагов генерации.
      max_seq_len: Рабочая длина входа модели.

    Returns:
      Список идентификаторов с учётом исходной подсказки и сгенерированных токенов.

    Raises:
      ValueError: Если подсказка пуста.
    """
    if len(prompt_ids) == 0:
        raise ValueError('prompt_ids не может быть пустым.')

    generated = list(prompt_ids)
    for _ in range(max_new_tokens):
        model_input = np.full((1, max_seq_len), PAD_ID, dtype=np.int32)
        clipped = generated[:max_seq_len]
        model_input[0, : len(clipped)] = clipped

        probs = model.predict(model_input, verbose=0)[0]
        query_index = min(len(clipped) - 1, max_seq_len - 1)
        next_token = int(np.argmax(probs[query_index]))
        generated.append(next_token)

        if next_token == EOS_ID or len(generated) >= max_seq_len:
            break
    return generated


transition = {
    TOKEN_A: TOKEN_B,
    TOKEN_B: TOKEN_C,
    TOKEN_C: TOKEN_A,
    TOKEN_D: TOKEN_B,
}

prompts = ([BOS_ID, TOKEN_A], [BOS_ID, TOKEN_B], [BOS_ID, TOKEN_C], [BOS_ID, TOKEN_D]) * 5
success_count = 0

for prompt in prompts:
    generated = generate_greedy(model, prompt, max_new_tokens=6)
    first_new = generated[len(prompt)]
    if first_new == transition[prompt[-1]]:
        success_count += 1

print(f'Корректных продолжений: {success_count} из {len(prompts)}')
assert success_count >= 18, 'Детерминированная генерация не достигла порога 18/20.'

for prompt in prompts[:4]:
    generated = generate_greedy(model, prompt, max_new_tokens=8)
    print('prompt:', ids_to_text(prompt), '| generated:', ids_to_text(generated))


## Шаг 6: диагностика внимания


In [ ]:
trace_token_ids = keras.Input(shape=(MAX_SEQ_LEN - 1,), dtype='int32', name='trace_token_ids')
trace_padding_mask = layers.Lambda(lambda t: tf.not_equal(t, PAD_ID), name='trace_padding_mask')(trace_token_ids)
trace_embeddings = embedding_layer(trace_token_ids)
_, trace_attention_scores = decoder_layer(
    trace_embeddings,
    padding_mask=trace_padding_mask,
    return_attention_scores=True,
)
trace_model = keras.Model(trace_token_ids, trace_attention_scores, name='attention_trace_model')

sample_tokens = X_test[:1]
attention_scores = trace_model.predict(sample_tokens, verbose=0)
mean_scores = attention_scores[0].mean(axis=0)

useful_len = int(np.sum(sample_tokens[0] != PAD_ID))
heatmap = mean_scores[:useful_len, :useful_len]

future_mass = float(np.triu(heatmap, k=1).sum())
total_mass = float(heatmap.sum())
future_ratio = future_mass / total_mass if total_mass > 0 else 0.0

print(f'Отношение массы внимания в будущем: {future_ratio:.8f}')
assert future_ratio < 1e-4, 'Обнаружено заметное внимание к будущим позициям.'

labels = [ID_TO_TOKEN[int(t)] for t in sample_tokens[0][:useful_len]]
plt.figure(figsize=(6, 5))
plt.imshow(heatmap, cmap='magma')
plt.colorbar()
plt.xticks(range(useful_len), labels, rotation=45)
plt.yticks(range(useful_len), labels)
plt.title('Средняя карта внимания без доступа в будущее')
plt.xlabel('Позиция ключа')
plt.ylabel('Позиция запроса')
plt.tight_layout()


## Чек-лист перед завершением
1. Все `TODO` закрыты.
2. Воспроизводимость данных подтверждена на первых `k` примерах.
3. Токенная точность и перплексия проходят пороги.
4. Детерминированная генерация выполняет порог `18/20`.
5. Карта внимания не содержит доступа к будущим позициям.
